# Pearls · AI voice (free GPU)

Makes the audio for Pearls' AI-translated chapters: one MP3 per section (`ai-BB-CC-SS.mp3`), 1,218 sections, about 70 hours.

1. **Runtime → Change runtime type → T4 GPU** (free).
2. Put `pearls-tts-kit.zip` in your Google Drive, in a folder `pearls-tts` (or upload it when step 1 asks).
3. Run the cells in order. Everything is saved to Drive as it goes, so when Colab disconnects (it will), reopen the notebook, run steps 1–3 again and step 5 carries on where it stopped.

Chatterbox is MIT-licensed (Resemble AI) and marks its audio with an inaudible watermark. Kokoro is Apache 2.0. Label the result as an AI voice in the app.

In [ ]:
# 1. Get the kit (from Drive, or upload it)
import os, glob, shutil
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/pearls-tts'
os.makedirs(WORK, exist_ok=True)
kit = sorted(glob.glob(WORK + '/pearls-tts-kit*.zip'))
if not kit:
    from google.colab import files
    for name in files.upload():
        shutil.move(name, WORK + '/' + name)
    kit = sorted(glob.glob(WORK + '/pearls-tts-kit*.zip'))
!unzip -oq "{kit[-1]}" -d /content
%cd /content/pearls-tts
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"


In [ ]:
# 2. Install (2-4 minutes). If Colab asks to restart the session, do it, then run step 1 and go on to step 3.
!pip -q install chatterbox-tts kokoro-onnx soundfile


In [ ]:
# 3. Choose the engine and voice
ENGINE = "chatterbox"        # "chatterbox", "chatterbox-turbo" (needs VOICE) or "kokoro"
VOICE = ""                   # Chatterbox: path to a 10-20 s WAV of a voice you have the right to use ("" = built-in voice)
                             # Kokoro: a voice name, e.g. "am_michael", "bf_emma", "am_adam", "bm_george"
OUT = f"{WORK}/audio-{ENGINE}"
voice_arg = f'--voice "{VOICE}"' if VOICE else ""
print("Saving to", OUT)


In [ ]:
# 4. Listen first: one section (Ḥalla, Kashrut ch. 11) and one from Berakhot
!python pearls_tts.py --engine {ENGINE} {voice_arg} --only 17-11 10-03 --sections 1 --out "{WORK}/samples-{ENGINE}"
from IPython.display import Audio, display
for f in sorted(glob.glob(f"{WORK}/samples-{ENGINE}/*.mp3")):
    print(os.path.basename(f)); display(Audio(f))


In [ ]:
# 5. Everything (resumes where it stopped). Keep the tab open; free sessions last a few hours.
#    To do one book first, add e.g.  --only 10   (Berakhot)
!python pearls_tts.py --engine {ENGINE} {voice_arg} --out "{OUT}"


In [ ]:
# 6. Progress
import json
m = json.load(open(f"{OUT}/manifest.json")) if os.path.exists(f"{OUT}/manifest.json") else {}
print(f"{len(m)} of 1218 sections, {sum(m.values())/3600:.1f} hours of audio")


In [ ]:
# 7. (Optional) one zip to download; the MP3s are already in Drive
!cd "{WORK}" && zip -qr "audio-{ENGINE}.zip" "audio-{ENGINE}" && ls -la "audio-{ENGINE}.zip"
